# SPICE microstructure products: analysis starter

A starting point for analyses that use the microstructure results from FKt260806 (VMP-250 and VMP-5500). It reads **only the exported products** (`docs/products.md`), never intermediate pipeline outputs, so it can be copied into a separate analysis repository unchanged. Point `SPICE_PRODUCTS` at the products folder.

Rules that keep results defensible (CLAUDE.md, `docs/decisions.md`):

* **Check `product_version`.** Products change only with a version bump and a dated entry in `docs/decisions.md`.
* **Never form diffusivities per window.** T_z and S_z approach zero. Average chi, epsilon and the gradients over windows and profiles first, then divide (section 7).
* **Average arithmetically over profiles; bootstrap over profiles** for intervals. Epsilon and chi are lognormal and intermittent.
* **Flags, not missing data.** Exclusions and cautions are flag variables; select with them explicitly.
* **K_S is a model result.** K_S = R_ρ K_T / γ assumes salt fingers carry the fluxes, and γ (0.5–0.7) is poorly constrained. Carry the γ range through.
* **Don't use K_ρ = 0.2 ε/N² in finger-favourable water** (the density flux is upgradient there).

Sections: 0 setup · 1 casts · 2 segments and QC · 3 mean profiles · 4 diffusivities · 5 finger attribution · 6 per-cast metrics · 7 your own bin averages (including 20–150 m) · 8 template for a tracer (nitrate) flux.

## 0. Setup and provenance

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

PRODUCTS = Path(os.environ.get("SPICE_PRODUCTS", "../outputs/recommended")).resolve()
EXPECTED_PRODUCT_VERSION = "0.5"       # update deliberately when you move to newer products
INSTRUMENTS = ["vmp250", "vmp5500"]
LABEL = {"vmp250": "VMP-250", "vmp5500": "VMP-5500"}
COLOR = {"vmp250": "#2a78d6", "vmp5500": "#eb6834"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(PRODUCTS)

In [ ]:
def load_products(inst: str) -> dict:
    p = {
        "casts": pd.read_csv(PRODUCTS / f"casts_{inst}.csv", parse_dates=["start_utc"]),
        "segments": xr.open_dataset(PRODUCTS / f"segments_{inst}.nc"),
        "binned": xr.open_dataset(PRODUCTS / f"binned_{inst}.nc"),
        "diffusivity": pd.read_csv(PRODUCTS / f"diffusivity_{inst}.csv", dtype={"depth_bin": str}),
        "attribution": pd.read_csv(PRODUCTS / f"attribution_{inst}.csv"),
        "cast_metrics": pd.read_csv(PRODUCTS / f"cast_metrics_{inst}.csv"),
    }
    v = p["segments"].attrs["product_version"]
    if v != EXPECTED_PRODUCT_VERSION:
        warnings.warn(f"{inst}: product_version {v}, expected {EXPECTED_PRODUCT_VERSION}; "
                      "read docs/decisions.md before comparing with earlier results")
    return p


P = {inst: load_products(inst) for inst in INSTRUMENTS}
keys = ["product_version", "run_name", "date_created", "git_commit", "config_sha256",
        "microstructure_tpw_commit", "instrument"]
pd.DataFrame({inst: {k: P[inst]["segments"].attrs.get(k) for k in keys} for inst in INSTRUMENTS})

Caveats carried in the products' metadata:

In [ ]:
for c in P["vmp5500"]["segments"].attrs["caveats"].split(" | "):
    print("*", c)

## 1. Casts

One row per profile: time, position, station (`process_station` or transect), exclusions, cautions, probe serials, Sealog notes. The 48 h process station is at 11.626 N, 50.577 W.

In [ ]:
casts = pd.concat([P[i]["casts"] for i in INSTRUMENTS], ignore_index=True)
casts["station"] = casts["station"].fillna("transect")
display(casts.groupby(["instrument", "station"]).size().rename("profiles").to_frame())

fig, ax = plt.subplots(figsize=(7, 5))
for inst in INSTRUMENTS:
    c = P[inst]["casts"]
    ax.scatter(c["lon"], c["lat"], s=14 if inst == "vmp250" else 30, color=COLOR[inst], label=LABEL[inst])
ax.scatter([-50.577], [11.626], s=300, facecolor="none", edgecolor="k", label="process station")
ax.set(xlabel="longitude (°E)", ylabel="latitude (°N)")
ax.legend();

## 2. Segments and QC

`segments_<inst>.nc`: one record per 8 s dissipation window (about 6 m tall, stepped about 3 m). It holds:

* epsilon per probe and combined, plus `epsilon_shear_bounded` = max(ε, noise floor), which is an upper bound where the probe is at its floor;
* chi per thermistor and combined, and `epsilon_T`;
* background hydrography on 10 m fits (`*_10m`);
* the Turner class and staircase labels (`staircase_vdb2021`, `staircase_rollo2022`), and Re_b;
* QC flags.

The selection used by the published averages is `shear_keep` for epsilon and `chi_keep` for chi. `segment_keep` (speed and vibration flags) is a stricter, optional cut; the published averages don't apply it (`segments.enabled: false`).

In [ ]:
def segments_df(inst: str) -> pd.DataFrame:
    d = P[inst]["segments"].to_dataframe().reset_index()
    d["instrument"] = inst
    d["cast"] = d["profile_file"].str.split("_prof").str[0]
    return d


def flag_counts(inst: str, var: str) -> pd.Series:
    # Windows with each bit of a QC flag variable set
    v = P[inst]["segments"][var]
    masks = [int(m) for m in str(v.attrs["flag_masks"]).strip("[]").split()]
    names = v.attrs["flag_meanings"].split()
    x = v.values.astype(int)
    return pd.Series({n: int(((x & m) > 0).sum()) for n, m in zip(names, masks)}, name=var)


S = {inst: segments_df(inst) for inst in INSTRUMENTS}
for inst in INSTRUMENTS:
    d = S[inst]
    print(f"{LABEL[inst]}: {len(d)} windows, {d['profile_file'].nunique()} profiles; "
          f"shear_keep {d['shear_keep'].mean():.1%}, chi_keep {d['chi_keep'].mean():.1%}, "
          f"shear at noise floor {d['shear_at_floor'].mean():.1%}")
pd.concat([flag_counts("vmp5500", v) for v in ["segment_flags", "shear_flags", "chi_flags"]], axis=1)

One profile from the process station: turbulence, temperature mixing, and the temperature staircase, coloured by staircase label.

In [ ]:
inst = "vmp5500"
d = S[inst]
station_profiles = P[inst]["casts"].query("station == 'process_station'")["profile_file"]
prof = station_profiles.iloc[0]
q = d[d["profile_file"] == prof].sort_values("depth")
lab_color = {"interface": "#c4501f", "layer": "#2a78d6", "straddles": "#a9a8a2", "none": "#d9d9d9"}

fig, axs = plt.subplots(1, 3, figsize=(12, 7), sharey=True)
e = q[q["shear_keep"] == 1]
axs[0].semilogx(e["epsilon_shear_bounded"], e["depth"], ".-", color=COLOR[inst], ms=3, lw=0.6)
axs[0].semilogx(e["epsilon_floor"], e["depth"], ":", color="gray", label="noise floor")
axs[0].set_xlabel("ε (W kg⁻¹)")
axs[0].legend()
c = q[q["chi_keep"] == 1]
axs[1].semilogx(c["chi"], c["depth"], ".-", color=COLOR[inst], ms=3, lw=0.6)
axs[1].set_xlabel("χ (K² s⁻¹)")
axs[2].plot(q["CT"], q["depth"], color="k", lw=0.6)
for lab, col in lab_color.items():
    s = q[q["staircase_rollo2022"] == lab]
    axs[2].scatter(s["CT"], s["depth"], s=8, color=col, label=lab, zorder=3)
axs[2].set_xlabel("Θ (°C)")
axs[2].legend(title="rollo2022")
axs[0].set_ylabel("depth (m)")
axs[0].invert_yaxis()
fig.suptitle(prof);

## 3. Mean profiles with bootstrap intervals

`binned_<inst>.nc` has per-profile 5 m bin means (dimensions `profile`, `depth`). Average arithmetically over profiles and resample profiles for the interval. Here the 5 m bins are first averaged to `bin_m` within each profile.

In [ ]:
def profile_mean(inst: str, var: str, bin_m: float = 25, n_boot: int = 1000, min_profiles: int = 3,
                 profiles=None, seed: int = 0) -> pd.DataFrame:
    x = P[inst]["binned"][var].to_pandas()                       # profile x depth (5 m)
    if profiles is not None:
        x = x[x.index.isin(profiles)]
    coarse = np.floor(x.columns.to_numpy() / bin_m) * bin_m + bin_m / 2
    per = x.T.groupby(coarse).mean().T                          # profile x coarse bin
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(per), size=(n_boot, len(per)))
    arr = per.to_numpy()
    with warnings.catch_warnings():                             # bins empty in a resample
        warnings.simplefilter("ignore", RuntimeWarning)
        boot = np.stack([np.nanmean(arr[i], axis=0) for i in idx])
        out = pd.DataFrame({"depth": per.columns, "mean": np.nanmean(arr, axis=0),
                            "ci_low": np.nanpercentile(boot, 2.5, axis=0),
                            "ci_high": np.nanpercentile(boot, 97.5, axis=0),
                            "n_profiles": per.notna().sum(axis=0).to_numpy()})
    return out[out["n_profiles"] >= min_profiles]


fig, axs = plt.subplots(1, 2, figsize=(11, 7), sharey=True)
for inst in INSTRUMENTS:
    for ax, var in zip(axs, ["epsilon_shear_bounded", "chi"]):
        m = profile_mean(inst, var)
        ax.fill_betweenx(m["depth"], m["ci_low"], m["ci_high"], color=COLOR[inst], alpha=0.2, lw=0)
        ax.plot(m["mean"], m["depth"], color=COLOR[inst], label=LABEL[inst])
axs[0].set(xscale="log", xlabel="ε, bounded at the noise floor (W kg⁻¹)", ylabel="depth (m)")
axs[1].set(xscale="log", xlabel="χ (K² s⁻¹)")
axs[0].invert_yaxis()
axs[0].legend();

## 4. Published diffusivities (150 dbar to the bottom)

`diffusivity_<inst>.csv` (and `.nc`): bin averages over windows and profiles, with 95% bootstrap intervals over profiles. Each row is one combination of:

* `instrument_group`: `<inst> all free fall`, `process_station free fall` or `transect free fall`;
* `method`: staircase labels, `vdb2021` or `rollo2022`;
* `turner`: `finger_any`, `strong_finger`, `control` (finger-favourable water outside staircases) or `all_turner`;
* `staircase`: `interface`, `layer`, `straddles`, `none` or `all`;
* `gradients`: `background` (10 m fits), `finescale` (1 m gradient variance) or `interface_local`;
* `depth_bin`: 25 m bin centre, or `pooled`.

`K_S_applicable` and `K_rho_valid` say where those quantities may be used.

In [ ]:
def diff(inst: str, group: str = "all free fall", method: str = "rollo2022", turner: str = "finger_any",
         staircase: str = "all", gradients: str = "finescale", pooled: bool = False) -> pd.DataFrame:
    t = P[inst]["diffusivity"]
    q = t[(t["instrument_group"] == f"{inst} {group}") & (t["method"] == method) & (t["turner"] == turner)
          & (t["staircase"] == staircase) & (t["gradients"] == gradients)]
    if pooled:
        return q[q["depth_bin"] == "pooled"]
    q = q[(q["depth_bin"] != "pooled") & q["min_profiles_ok"]].copy()
    q["depth"] = q["depth_bin"].astype(float)
    return q.sort_values("depth")


cols = ["staircase", "n_profiles", "K_T", "K_T_ci_low", "K_T_ci_high", "K_S_g0.65", "K_S_eps_g0.65", "K_GM"]
pd.concat([pd.concat([diff(i, staircase=s, pooled=True) for s in ["all", "interface", "layer", "none"]])
             .assign(instrument=i)[["instrument"] + cols] for i in INSTRUMENTS]).round(8)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 7))
for ax, inst in zip(axs, INSTRUMENTS):
    q = diff(inst)
    ax.fill_betweenx(q["depth"], q["K_T_ci_low"], q["K_T_ci_high"], color="#2a78d6", alpha=0.2, lw=0)
    ax.plot(q["K_T"], q["depth"], color="#2a78d6", label="K_T")
    ks = q[["K_S_g0.5", "K_S_g0.65", "K_S_g0.7"]]
    ax.fill_betweenx(q["depth"], ks.min(axis=1), ks.max(axis=1), color="#eb6834", alpha=0.25, lw=0,
                     label="K_S, γ 0.5–0.7")
    ax.plot(q["K_S_g0.65"], q["depth"], color="#eb6834", label="K_S (γ 0.65)")
    ax.plot(q["K_GM"], q["depth"], "k-.", lw=1, label="Garrett–Munk")
    ax.set(xscale="log", xlim=(1e-7, 1e-3), xlabel="diffusivity (m² s⁻¹)", title=LABEL[inst])
    ax.invert_yaxis()
axs[0].set_ylabel("depth (m)")
axs[1].legend();

## 5. Finger attribution ladder

E = ⟨χ_obs⟩ / ⟨χ_expected⟩ against three baselines:

* Γ = 0.2 theory;
* Garrett–Munk;
* `nonstaircase_control`: finger-favourable water outside staircases at the same depths, matched in Re_b. This gives a lower bound.

The finger fraction is 1 − 1/E. **The spread across baselines is part of the result**, so report it as a range.

In [ ]:
att = pd.concat([P[i]["attribution"] for i in INSTRUMENTS])
att = att[att["instrument_group"].str.endswith("all free fall") & (att["method"] == "rollo2022")
          & (att["gradients"] == "finescale")]
att.pivot_table(index=["instrument_group", "staircase"], columns="baseline", values="E").round(2)

## 6. Per-cast metrics (for mesoscale comparisons)

One row per profile: salinity maximum, median R_ρ, staircase coverage, interface counts and jumps, K_T in finger water and its ratio to GM, mean ε and χ.

The two instruments cover different depth ranges (the VMP-250 reaches about 290 m, the VMP-5500 1000 m), so don't pool depth-dependent metrics such as staircase coverage across them.

In [ ]:
cm = pd.concat([P[i]["cast_metrics"] for i in INSTRUMENTS], ignore_index=True)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for inst in INSTRUMENTS:
    c = cm[cm["instrument"] == inst]
    axs[0].scatter(c["R_rho_median"], c["K_T_finger_over_GM"], color=COLOR[inst], label=LABEL[inst], s=18)
    axs[1].scatter(c["SA_max"], c["staircase_fraction_rollo2022"], color=COLOR[inst], label=LABEL[inst], s=18)
axs[0].set(xlabel="median R_ρ (150 m to bottom)", ylabel="K_T / K_GM, finger-favourable water", yscale="log")
axs[1].set(xlabel="salinity maximum (g kg⁻¹)", ylabel="staircase fraction (rollo2022)")
axs[0].legend();

## 7. Your own bin averages, including 20–150 m

The published diffusivities start at 150 dbar (the staircase tests). For depth ranges or groupings they don't cover, form averages from the segments **the same way**:

1. sum χ, T_z², αT_z, βS_z, ε and N² over the windows of each profile in a bin;
2. sum over profiles;
3. divide;
4. bootstrap by resampling profiles.

The first cell checks this reproduces a published bin exactly.

Limitation: only the **background** (10 m fit) gradients are in the segments product. The published `finescale` K_T uses 1 m gradient variance, which isn't exported, so this function can't reproduce that variant.

In [ ]:
def window_selection(d: pd.DataFrame) -> pd.Series:
    # Windows used by the published averages (dd_test): both QC keeps, N2 > 0, valid salinity
    return ((d["shear_keep"] == 1) & (d["chi_keep"] == 1) & (d["N2_10m"] > 0)
            & np.isfinite(d["T_z_10m"]) & np.isfinite(d["S_z_10m"]) & np.isfinite(d["alpha"])
            & np.isfinite(d["epsilon_shear_bounded"]) & np.isfinite(d["chi"])
            & (d["ts_salinity_invalid"] != 1))


def _estimates(s: np.ndarray, gammas) -> dict:
    chi, tz2, atz, bsz, eps, n2, n = s
    kt, rr = chi / (2 * tz2), (atz / bsz if bsz > 0 else np.nan)
    out = {"K_T": kt, "R_rho": rr, "K_rho": 0.2 * eps / n2, "eps_mean": eps / n, "chi_mean": chi / n}
    for g in gammas:
        out[f"K_S_g{g}"] = rr * kt / g if rr > 1 else np.nan
    return out


def bin_diffusivity(d: pd.DataFrame, edges, gammas=(0.5, 0.65, 0.7), n_boot: int = 1000, seed: int = 0,
                    min_profiles: int = 3) -> pd.DataFrame:
    # Diffusivities from window sums per profile, in pressure bins (edges in dbar)
    d = d[window_selection(d)].assign(
        tz2=lambda x: x["T_z_10m"] ** 2, atz=lambda x: x["alpha"] * x["T_z_10m"],
        bsz=lambda x: x["beta"] * x["S_z_10m"], eps=lambda x: x["epsilon_shear_bounded"],
        finger=lambda x: x["turner_class"].isin(["finger", "strong_finger"]))
    d["bin"] = pd.cut(d["pressure"], edges, right=False)
    rng = np.random.default_rng(seed)
    cols = ["chi", "tz2", "atz", "bsz", "eps", "N2_10m"]
    rows = []
    for b, db in d.groupby("bin", observed=True):
        per = db.groupby("profile_file")[cols].sum()
        per["n"] = db.groupby("profile_file").size()
        if len(per) < min_profiles:
            continue
        arr = per.to_numpy()
        r = {"P_low": b.left, "P_high": b.right, "n_windows": len(db), "n_profiles": len(per),
             "finger_fraction": db["finger"].mean(), **_estimates(arr.sum(axis=0), gammas)}
        boot = [_estimates(arr[i].sum(axis=0), gammas)["K_T"] for i in rng.integers(0, len(per), (n_boot, len(per)))]
        r["K_T_ci_low"], r["K_T_ci_high"] = np.nanpercentile(boot, [2.5, 97.5])
        rows.append(r)
    return pd.DataFrame(rows)


# Check: reproduce one published bin (VMP-5500, all water, background gradients, 400-425 dbar)
mine = bin_diffusivity(S["vmp5500"], [400, 425]).iloc[0]
pub = diff("vmp5500", turner="all_turner", gradients="background").query("depth_bin == '412.5'").iloc[0]
print(f"K_T  mine {mine['K_T']:.4g}   published {pub['K_T']:.4g}")
print(f"R_rho mine {mine['R_rho']:.4g}   published {pub['R_rho']:.4g}")
assert np.isclose(mine["K_T"], pub["K_T"], rtol=1e-6) and mine["n_windows"] == pub["n_windows"]

The upper ocean (20–150 dbar) is mostly not finger-favourable; `finger_fraction` in each bin shows it. Where it's low, K_ρ = 0.2⟨ε⟩/⟨N²⟩ is the standard estimate, and K_T is the independent one from χ. **K_S = R_ρ K_T/γ applies only where the water is finger-favourable.** Where K_ρ sits near the noise floor, ε is an upper bound (`epsilon_shear_bounded`).

In [ ]:
edges = np.arange(20, 300 + 1, 10)
upper = {inst: bin_diffusivity(S[inst], edges) for inst in INSTRUMENTS}

fig, axs = plt.subplots(1, 2, figsize=(12, 7), sharey=True)
for ax, inst in zip(axs, INSTRUMENTS):
    u = upper[inst]
    z = (u["P_low"].astype(float) + u["P_high"].astype(float)) / 2
    ax.fill_betweenx(z, u["K_T_ci_low"], u["K_T_ci_high"], color="#2a78d6", alpha=0.2, lw=0)
    ax.plot(u["K_T"], z, color="#2a78d6", label="K_T (background gradients)")
    ax.plot(u["K_rho"], z, color="gray", ls=":", label="K_ρ = 0.2⟨ε⟩/⟨N²⟩")
    fz = u["finger_fraction"] > 0.5
    ax.plot(u["K_S_g0.65"].where(fz), z, color="#eb6834", label="K_S (γ 0.65), finger-favourable bins")
    ax.axhline(150, color="k", lw=0.8, ls="--")
    ax.set(xscale="log", xlabel="diffusivity (m² s⁻¹)", title=LABEL[inst])
axs[0].set_ylabel("pressure (dbar)")
axs[0].invert_yaxis()
axs[1].legend(fontsize=8)
upper["vmp5500"].round(8).head(15)

## 8. Template: vertical tracer flux (e.g. nitrate)

F = −K ∂C/∂z (z up), or with depth positive down, the **upward** flux is F_up = K ∂C/∂(depth). In finger-favourable water nitrate behaves like salt (low molecular diffusivity), so use K_S with its γ range; elsewhere use K_ρ or K_T.

Your tracer data don't come into this repository. Load them in the analysis repository, interpolate the gradient onto the diffusivity depths, and keep the K uncertainty (bootstrap interval and γ range) attached.

In [ ]:
def upward_flux(k_depth, k_value, c_depth, c_value):
    # Upward flux K dC/d(depth) at the diffusivity depths; units of K times C per m
    # (e.g. m2 s-1 x mmol m-3 / m = mmol m-2 s-1). Positive = upward.
    c_depth, c_value = np.asarray(c_depth, float), np.asarray(c_value, float)
    order = np.argsort(c_depth)
    dcdz = np.gradient(c_value[order], c_depth[order])
    return np.asarray(k_value, float) * np.interp(k_depth, c_depth[order], dcdz, left=np.nan, right=np.nan)


# Example, once you have a nitrate profile (depth in m, NO3 in mmol m-3):
# no3 = pd.read_csv("path/to/nitrate_profile.csv")
# q = diff("vmp5500", group="process_station free fall")
# for g in ["0.5", "0.65", "0.7"]:
#     q[f"F_NO3_g{g}"] = upward_flux(q["depth"], q[f"K_S_g{g}"], no3["depth"], no3["NO3"]) * 86400   # mmol m-2 d-1

## Notes

* Methods and their reasons: `docs/decisions.md`. Variable definitions: `docs/products.md`. Rerun the products: `python src/run_pipeline.py --config config/config.yaml --sensitivity fft2_diss8_mle`.
* Largest processing systematics on K_S: the FP07 response model (about 2×) and γ (1.4×). Sensitivity runs are summarised in `sensitivity_summary.csv` in the products folder.
* Staircase labels: `rollo2022` (tuned) is used above; `vdb2021` (published thresholds) is the alternative and labels fewer windows.